In [1]:
import torch
from torch import nn
import torch.nn.functional as F

In [ ]:
# First test on an example before implementing the whole block

# B, C, H, W
x = torch.arange(120).reshape(2, 4, 3, 5)
# B, C, N where N is HxW
flat = torch.flatten(x, start_dim=-2)
# B, N, C
tokens = torch.swapaxes(flat, -1, -2)

print("Shape of x:")
print(x.shape)
print("\n")

print("Shape of tokens:")
print(tokens.shape)
print("\n")

# Verify
print(x[0, :, 1, 2])
print(torch.equal(x[0, :, 1, 2], tokens[0, 7]))
print("\n")

# Recovering
tokens_axes_swapped = torch.swapaxes(tokens, -1, -2)
print("Shape of tokens_axes_swapped:")
print(tokens_axes_swapped.shape)
print("\n")

x_recovered = tokens_axes_swapped.reshape(x.shape)
print("Shape of x_recovered:")
print(x_recovered.shape)
print(torch.equal(x_recovered, x))

Shape of x:
torch.Size([2, 4, 3, 5])


Shape of tokens:
torch.Size([2, 15, 4])


tensor([ 7, 22, 37, 52])
True


Shape of tokens_axes_swapped:
torch.Size([2, 4, 15])


Shape of x_recovered:
torch.Size([2, 4, 3, 5])
True


In [ ]:
class AttentionBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.channels = channels

        # GroupNorm, 8 groups, over channels
        self.norm = nn.GroupNorm(num_groups=8, num_channels=channels)

        # Projections W_q, W_k, W_v
        self.q_proj = nn.Linear(channels, channels)
        self.k_proj = nn.Linear(channels, channels)
        self.v_proj = nn.Linear(channels, channels)
        
        self.output_proj = nn.Linear(channels, channels)

        # Set the weights and bias of output_proj to zero 
        nn.init.zeros_(self.output_proj.weight)
        nn.init.zeros_(self.output_proj.bias)

    def forward(self, x):
        # input x : (B, C, H, W)
        # Normalize
        x_normalized = self.norm(x) # (B, C, H, W)
        # Flatten
        x_flat = torch.flatten(x_normalized, start_dim=-2) # (B, C, N), where N = H x W
        # Swap axes and get the tokens we will use
        tokens = torch.swapaxes(x_flat, -1, -2) # (B, N, C)

        # Apply the projections to tokens, and get q, k, v.
        q = self.q_proj(tokens) # (B, N, C)
        k = self.k_proj(tokens) # (B, N, C)
        v = self.v_proj(tokens) # (B, N, C)

        scores = (q @ torch.swapaxes(k, -1, -2)) / (self.channels ** 0.5) # (B, N, N)
        weights = F.softmax(scores, dim=2) # (B, N, N)
        attn_out = weights @ v # (B, N, C)
        projected = self.output_proj(attn_out) # (B, N, C)
        swapped_back = torch.swapaxes(projected, -1, -2) # (B, C, N)
        unflattened = swapped_back.reshape(x.shape) # (B, C, H, W)

        return x + unflattened
